## Setup

[Link to Question](https://leetcode.com/problems/game-play-analysis-iv/description/?envType=study-plan-v2&envId=top-sql-50)



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import date

spark = SparkSession.builder.appName("GamePlayAnalysisIV").getOrCreate()

# Schema
schema = StructType([
    StructField("player_id", IntegerType(), True),
    StructField("device_id", IntegerType(), True),
    StructField("event_date", DateType(), True),
    StructField("games_played", IntegerType(), True),
])

# Sample Data
data = [
    (1, 2, date(2016, 3, 1), 5),
    (1, 2, date(2016, 3, 2), 6),  # logged in day after first login
    (2, 3, date(2017, 6, 25), 1),
    (3, 1, date(2016, 3, 2), 0),
    (3, 4, date(2018, 7, 3), 5),
]

activity_df = spark.createDataFrame(data, schema)
activity_df.createOrReplaceTempView("Activity")
activity_df.show()

+---------+---------+----------+------------+
|player_id|device_id|event_date|games_played|
+---------+---------+----------+------------+
|        1|        2|2016-03-01|           5|
|        1|        2|2016-03-02|           6|
|        2|        3|2017-06-25|           1|
|        3|        1|2016-03-02|           0|
|        3|        4|2018-07-03|           5|
+---------+---------+----------+------------+



## Code

In [11]:
from pyspark.sql import functions as F

min_date_df = activity_df.groupBy(F.col("player_id")).agg(F.min(F.col("event_date")).alias("min_date"))

distinct_count = activity_df.select(F.col("player_id")).distinct().count()
matched_count = activity_df.join(min_date_df, "player_id").filter(F.col("event_date") == F.date_add(F.col("min_date"), 1)).select("player_id").distinct().count()

spark.createDataFrame([(round(matched_count / distinct_count, 2),)], ["fraction"]).show()

+--------+
|fraction|
+--------+
|    0.33|
+--------+

